_We'll build this together, live. This session's main exercises covered sorting; this application digs into recursion, the technique behind the merge sort you saw in the Further Exercises._

## Application: Analysing Overlap in a Managed Portfolio

<img src="images/overlap.png" alt="Article: what is mutual fund overlap and why it could be a concern for investors" width="500">

A portfolio manager has inherited a complex portfolio that consists of both direct stock holdings and indirect holdings through various investment vehicles. Before considering changes, they need to understand where potential overlaps exist.

- **Direct stock holdings**: e.g. `META` held directly
- **Exchange-Traded Funds (ETFs)**: baskets of stocks, e.g. an ETF holding `META`
- **Fund of Funds (FoFs)**: funds that invest in multiple other funds, some of which hold `META`

This can be represented as a **nested list**: a list that can contain strings, or other lists, to any depth:
```python
managed_portfolio = [
    "META",   # a direct holding
    "GOOGL",  # a direct holding
    ["META", "GOOGL", "AMZN", "NFLX"],  # an ETF
    [
        ["META", "AMZN", "NFLX"],   # a fund within a FoF
        ["GOOGL", "AMZN", "NFLX"]   # another fund within the FoF
    ]
]
```

We want a **generic** solution to count how many times a given stock appears in a managed portfolio, however deeply it's nested. We'll build up to it in three stages: no nesting, one level of nesting, then any depth of nesting.

## Part 1: A flat portfolio

Let's start simple: assume the portfolio isn't nested at all, just a flat list of individual stocks.

A portfolio can be a flat list of stocks:

In [ ]:
flat_portfolio = ["META", "AMZN", "NFLX"]

No nesting here: just three stocks in a list.

**Write `count_occurrences_flat`** using a plain `for` loop: check each item against `target`, and count the matches.

In [ ]:
def count_occurrences_flat(data, target):
    """Count how many times target appears in a flat (non-nested) portfolio."""
    # your code here



**Test it:**

In [ ]:
portfolio_flat = ["META", "AMZN", "NFLX", "META", "GOOGL", "META"]
stock = "META"
overlap = count_occurrences_flat(portfolio_flat, stock)
print(f"{stock} is present {overlap} times in this portfolio.")

## Part 2: One level of nesting (ETFs)

Now suppose the portfolio can also hold **ETFs**: a list nested one level deep. Assume there are no FoFs yet, so a list never contains another list, just stocks. We can still solve this without recursion: just check whether each item is itself a list.

A nested example: one direct stock, plus one ETF (a list of stocks):

In [ ]:
nested_portfolio = ["NFLX", ["META", "AMZN"]]

**Hint:** to check whether something is a list, compare its `type` to `list`:

In [ ]:
type([1, 2, 3]) == list

In [ ]:
type("META") == list

**Write `count_occurrences_one_level`.** Loop through `data`; if an item is a list (an ETF), loop through it too; otherwise, it's a direct stock.

In [ ]:
def count_occurrences_one_level(data, target):
    """Count occurrences, assuming at most one level of nesting (ETFs, no FoFs)."""
    # your code here



**Test it:**

In [ ]:
portfolio_one_level = ["META", ["META", "GOOGL", "AMZN"], "NFLX", ["META", "NFLX"]]
stock = "META"
overlap = count_occurrences_one_level(portfolio_one_level, stock)
print(f"{stock} is present {overlap} times in this portfolio.")

## Part 3: Arbitrary nesting (FoFs too)

Real portfolios can nest arbitrarily deep: FoFs holding funds, which could themselves hold further funds. The one-level check from Part 2 breaks down here, since we don't know in advance how many levels to check. This is exactly what **recursion** solves: a function that calls itself for every level of nesting, however many there are.

**Step 1: the recursive case.** If `data` is a **list**, loop through its items and call `count_occurrences` again on each one: the function calling itself is what lets us handle *any* depth of nesting.

In [ ]:
def count_occurrences(data, target):
    """Count how many times target appears in a (possibly nested) portfolio."""
    # your code here



**Step 2: the base case.** Once `data` is no longer a list, it's a single stock: we've drilled all the way down. Check whether it matches `target`.

In [ ]:
def count_occurrences(data, target):
    """Count how many times target appears in a (possibly nested) portfolio."""
    # your code here



In [ ]:
managed_portfolio = [
    "META",   # a direct holding
    "GOOGL",  # a direct holding
    ["META", "GOOGL", "AMZN", "NFLX"],  # an ETF
    [
        ["META", "AMZN", "NFLX"],   # a fund within a FoF
        ["GOOGL", "AMZN", "NFLX"]   # another fund within the FoF
    ]
]

**Test it:**

In [ ]:
stock = "META"
overlap = count_occurrences(managed_portfolio, stock)
print(f"{stock} is present {overlap} times in this portfolio.")

## Part 4: Your turn: total portfolio value

Counting matches for one stock is useful, but suppose instead the fund-of-funds structure reports dollar **values**, not tickers: each sub-fund's allocation nested the same way:

```python
fund_of_funds_value = [
    [1_500_000, 2_300_000],          # sub-fund A's two holdings
    [800_000, [400_000, 250_000]],   # sub-fund B, itself holding a further nested fund
    650_000                          # a direct allocation
]
```

Write a function `total_value(data)` that returns the **total dollar value** held anywhere in this (possibly nested) structure.

It follows the same recursive shape as `count_occurrences`:
- **Recursive case**: if `data` is a list, sum up the results of calling `total_value` on each item.
- **Base case**: otherwise, `data` is a single dollar amount: return it directly.

In [ ]:
# your code here

## Part 5: Your turn: every stock at once

So far we've counted one stock at a time. Now write a function that returns a dictionary mapping **every** stock ticker in the portfolio to how many times it appears, however deeply nested. Try it two ways.

**Way 1: reuse what you've built.** Write two functions:
- `find_all_stocks(data)`: returns the **set** of every distinct stock ticker in `data`. Same recursive shape as `count_occurrences`: if `data` is a list, take the union (`|`) of the results on each item; otherwise, return a set containing just that one stock.
- `count_all_stocks(data)`: loops over `find_all_stocks(data)` and uses `count_occurrences` to build a dictionary mapping each stock to its count.

In [ ]:
def find_all_stocks(data):
    """Return the set of every distinct stock ticker in a (possibly nested) portfolio."""
    # your code here


def count_all_stocks(data):
    """Return a dictionary mapping every stock ticker to how many times it appears."""
    # your code here



**Way 2: a single function, with a scoped dictionary.** Write `count_all_stocks_v2(data)` as one stand-alone function. Inside it:
- define an empty dictionary to hold the counts;
- define an inner function that recursively walks `data`, updating that dictionary directly (it doesn't need to return anything: it just mutates the dictionary defined in the enclosing scope);
- call the inner function once on `data`, then return the dictionary.

In [ ]:
def count_all_stocks_v2(data):
    """Return a dictionary mapping every stock ticker to how many times it appears."""
    # your code here



**Test both:**

In [ ]:
print(count_all_stocks(managed_portfolio))
print(count_all_stocks_v2(managed_portfolio))